# Deep Learning sur données tabulaires, images et texte

## Objectif
Comparer trois architectures adaptées à trois types de données :
- un réseau dense pour les données tabulaires ;
- un CNN pour les images ;
- un réseau récurrent pour le texte.

Pour chaque expérience, les données sont séparées avant tout
prétraitement afin d’éviter les fuites de données.

In [2]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("Environnement prêt")

TensorFlow : 2.21.0
Environnement prêt


# 1. Classification sur données tabulaires : Titanic

## Objectif

Prédire si un passager a survécu à partir de caractéristiques
personnelles et liées au voyage.

Un réseau dense, ou MLP, convient car chaque passager est représenté
par une ligne contenant plusieurs variables numériques et catégorielles.

In [3]:
titanic = sns.load_dataset("titanic")

colonnes = [
    "survived",
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
]

df = titanic[colonnes].copy()

print("Dimensions :", df.shape)
df.head()

Dimensions : (891, 8)


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [4]:
print("Valeurs manquantes :")
display(df.isna().sum())

print("Répartition de la cible :")
display(df["survived"].value_counts(normalize=True))

Valeurs manquantes :


survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

Répartition de la cible :


survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="survived")
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train,
)

print("Entraînement :", X_train.shape)
print("Validation    :", X_val.shape)
print("Test          :", X_test.shape)

Entraînement : (569, 7)
Validation    : (143, 7)
Test          : (179, 7)


## Prétraitement des données

Les variables numériques et catégorielles nécessitent des traitements
différents.

### Variables numériques

- Les valeurs manquantes sont remplacées par la médiane.
- Les valeurs sont standardisées pour faciliter l’entraînement du réseau.

### Variables catégorielles

- Les valeurs manquantes sont remplacées par la modalité la plus fréquente.
- Chaque catégorie est transformée en colonne binaire avec un encodage one-hot.

La classe du billet `pclass` est traitée comme une catégorie. Le
préprocesseur est ajusté uniquement sur l’ensemble d’entraînement.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colonnes_numeriques = [
    "age",
    "sibsp",
    "parch",
    "fare",
]

colonnes_categorielles = [
    "pclass",
    "sex",
    "embarked",
]

pipeline_numerique = Pipeline(
    steps=[
        ("imputation", SimpleImputer(strategy="median")),
        ("normalisation", StandardScaler()),
    ]
)

pipeline_categoriel = Pipeline(
    steps=[
        ("imputation", SimpleImputer(strategy="most_frequent")),
        (
            "encodage",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

In [7]:
preprocesseur = ColumnTransformer(
    transformers=[
        (
            "numerique",
            pipeline_numerique,
            colonnes_numeriques,
        ),
        (
            "categoriel",
            pipeline_categoriel,
            colonnes_categorielles,
        ),
    ],
    verbose_feature_names_out=False,
)

In [8]:
X_train_prepare = preprocesseur.fit_transform(X_train)

X_val_prepare = preprocesseur.transform(X_val)
X_test_prepare = preprocesseur.transform(X_test)

In [9]:
X_train_prepare = X_train_prepare.astype("float32")
X_val_prepare = X_val_prepare.astype("float32")
X_test_prepare = X_test_prepare.astype("float32")

y_train_array = y_train.to_numpy(dtype="float32")
y_val_array = y_val.to_numpy(dtype="float32")
y_test_array = y_test.to_numpy(dtype="float32")

In [10]:
noms_variables = preprocesseur.get_feature_names_out()

print("Variables produites :")
print(noms_variables)

print("\nDimensions :")
print("Entraînement :", X_train_prepare.shape)
print("Validation    :", X_val_prepare.shape)
print("Test          :", X_test_prepare.shape)

print("\nValeurs manquantes après prétraitement :")
print("Entraînement :", np.isnan(X_train_prepare).sum())
print("Validation    :", np.isnan(X_val_prepare).sum())
print("Test          :", np.isnan(X_test_prepare).sum())

Variables produites :
['age' 'sibsp' 'parch' 'fare' 'pclass_1' 'pclass_2' 'pclass_3'
 'sex_female' 'sex_male' 'embarked_C' 'embarked_Q' 'embarked_S']

Dimensions :
Entraînement : (569, 12)
Validation    : (143, 12)
Test          : (179, 12)

Valeurs manquantes après prétraitement :
Entraînement : 0
Validation    : 0
Test          : 0


In [11]:
df_train_prepare = pd.DataFrame(
    X_train_prepare,
    columns=noms_variables,
    index=X_train.index,
)

df_train_prepare.head()

,age,sibsp,parch,fare,pclass_1,pclass_2,pclass_3,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
242,-0.064840,-0.462989,-0.461079,-0.453710,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
450,0.473657,0.481243,1.860636,-0.071871,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
721,-0.987978,0.481243,-0.461079,-0.529985,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
733,-0.526409,-0.462989,-0.461079,-0.398371,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
191,-0.834122,-0.462989,-0.461079,-0.398371,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


## Modèle de référence

La classe majoritaire correspond aux passagers qui n’ont pas survécu.
Un modèle utile doit faire mieux qu’une prédiction constante de cette
classe.

Cette référence empêche de considérer une accuracy apparemment élevée
comme suffisante.

In [12]:
classe_majoritaire = y_train.mode()[0]

accuracy_reference = (
    y_val == classe_majoritaire
).mean()

print("Classe toujours prédite :", classe_majoritaire)
print(f"Accuracy de référence : {accuracy_reference:.3f}")

Classe toujours prédite : 0
Accuracy de référence : 0.615


## Architecture du MLP

Le réseau reçoit les 12 variables produites par le préprocesseur.

- Une première couche dense de 16 neurones apprend des combinaisons
  entre les caractéristiques.
- La fonction ReLU introduit de la non-linéarité.
- Une couche Dropout désactive aléatoirement 20 % des activations pendant
  l’entraînement afin de limiter le surapprentissage.
- Une seconde couche dense de 8 neurones construit une représentation
  plus compacte.
- Le neurone de sortie utilise une sigmoïde et produit une probabilité
  de survie comprise entre 0 et 1.

Le réseau reste volontairement petit, car le jeu d’entraînement ne
contient que 569 passagers.

In [13]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

nombre_variables = X_train_prepare.shape[1]

modele_titanic = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(nombre_variables,),
            name="caracteristiques_passager",
        ),
        tf.keras.layers.Dense(
            16,
            activation="relu",
            name="couche_dense_1",
        ),
        tf.keras.layers.Dropout(
            0.20,
            name="dropout",
        ),
        tf.keras.layers.Dense(
            8,
            activation="relu",
            name="couche_dense_2",
        ),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="probabilite_survie",
        ),
    ],
    name="mlp_titanic",
)

In [14]:
modele_titanic.summary()

Model: "mlp_titanic"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ couche_dense_1 (Dense)               │ (None, 16)                  │             208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ couche_dense_2 (Dense)               │ (None, 8)                   │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ probabilite_survie (Dense)           │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 353 (1.38 KB)

 Trainable params: 353 (1.38 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
parametres_couche_1 = (12 * 16) + 16
parametres_couche_2 = (16 * 8) + 8
parametres_sortie = (8 * 1) + 1

total_calcule = (
    parametres_couche_1
    + parametres_couche_2
    + parametres_sortie
)

print("Couche 1 :", parametres_couche_1)
print("Couche 2 :", parametres_couche_2)
print("Sortie   :", parametres_sortie)
print("Total    :", total_calcule)

assert total_calcule == modele_titanic.count_params()

Couche 1 : 208
Couche 2 : 136
Sortie   : 9
Total    : 353


In [16]:
modele_titanic.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001,
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
        ),
        tf.keras.metrics.Precision(
            name="precision",
        ),
        tf.keras.metrics.Recall(
            name="recall",
        ),
        tf.keras.metrics.AUC(
            name="auc",
        ),
    ],
)